In [72]:
import pandas as pd

data = pd.read_csv('/content/email.csv')


In [73]:
print(data)

            Category                                            Message
0                ham  Go until jurong point, crazy.. Available only ...
1                ham                      Ok lar... Joking wif u oni...
2               spam  Free entry in 2 a wkly comp to win FA Cup fina...
3                ham  U dun say so early hor... U c already then say...
4                ham  Nah I don't think he goes to usf, he lives aro...
...              ...                                                ...
5568             ham               Will ü b going to esplanade fr home?
5569             ham  Pity, * was in mood for that. So...any other s...
5570             ham  The guy did some bitching but I acted like i'd...
5571             ham                         Rofl. Its true to its name
5572  {"mode":"full"                                    isActive:false}

[5573 rows x 2 columns]


In [74]:
X = data['Message']
y = data['Category']

In [11]:
import nltk
nltk.download()

NLTK Downloader
---------------------------------------------------------------------------
    d) Download   l) List    u) Update   c) Config   h) Help   q) Quit
---------------------------------------------------------------------------
Downloader> d

Download which package (l=list; x=cancel)?
  Identifier> l
Packages:
  [ ] abc................. Australian Broadcasting Commission 2006
  [ ] alpino.............. Alpino Dutch Treebank
  [ ] averaged_perceptron_tagger Averaged Perceptron Tagger
  [ ] averaged_perceptron_tagger_eng Averaged Perceptron Tagger (JSON)
  [ ] averaged_perceptron_tagger_ru Averaged Perceptron Tagger (Russian)
  [ ] averaged_perceptron_tagger_rus Averaged Perceptron Tagger (Russian)
  [ ] basque_grammars..... Grammars for Basque
  [ ] bcp47............... BCP-47 Language Tags
  [ ] biocreative_ppi..... BioCreAtIvE (Critical Assessment of Information
                           Extraction Systems in Biology)
  [ ] bllip_wsj_no_aux.... BLLIP Parser: WSJ Model
  [ 

True

In [63]:
import re
def clean_single_message(text):
    text = re.sub(r'https?://\S+|www\.\S+', '', text)#URLs
    text = re.sub(r'[^\w\s]', '', text)#Punctuation & Symbols
    text = re.sub(r'\d+', '', text)#Numbers
    text = re.sub(r'\s+', ' ', text).strip()#Whitespace Trimming
    return text

X_clean = [clean_single_message(msg) for msg in X]

In [75]:
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize

words = [word_tokenize(message) for message in X_clean]


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [76]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = list(set(stopwords.words('english')))
filtered_words = [word for word in words if not word in stop_words]


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [77]:
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
lemmas = [[lemmatizer.lemmatize(w) for w in message] for message in filtered_words]

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [78]:
messages = [' '.join(words) for words in lemmas]

print(messages)

['Go until jurong point crazy Available only in bugis n great world la e buffet Cine there got amore wat', 'Ok lar Joking wif u oni', 'Free entry in a wkly comp to win FA Cup final tkts st May Text FA to to receive entry questionstd txt rateTCs apply over', 'U dun say so early hor U c already then say', 'Nah I dont think he go to usf he life around here though', 'FreeMsg Hey there darling it been week now and no word back Id like some fun you up for it still Tb ok XxX std chgs to send to rcv', 'Even my brother is not like to speak with me They treat me like aid patent', 'As per your request Melle Melle Oru Minnaminunginte Nurungu Vettam ha been set a your callertune for all Callers Press to copy your friend Callertune', 'WINNER As a valued network customer you have been selected to receivea prize reward To claim call Claim code KL Valid hour only', 'Had your mobile month or more U R entitled to Update to the latest colour mobile with camera for Free Call The Mobile Update Co FREE on', 

In [79]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(ngram_range=(1,2))
X = cv.fit_transform(messages).toarray()


In [90]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

In [93]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score


num_ham = np.sum(y_train == 0)
num_spam = np.sum(y_train == 1)
scale_weight = num_ham / num_spam

model = XGBClassifier(
    scale_pos_weight=scale_weight,
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    eval_metric='logloss'
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [18:06:43] WARNING: /__w/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Accuracy: 0.9739910313901345


In [92]:
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import classification_report, accuracy_score

model = ComplementNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.9228699551569507

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.92      0.95       958
           1       0.89      0.94      0.91       157
           2       0.00      0.00      0.00         0

    accuracy                           0.92      1115
   macro avg       0.63      0.62      0.62      1115
weighted avg       0.98      0.92      0.95      1115



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
